# CIFAR-10 Image Classification using CNN

**Name:** Palak Narang
**Reg No:** 23BCE11819
**Objective:** Build a Convolutional Neural Network (CNN) to classify images from the CIFAR-10 dataset into 10 categories with a target accuracy of 85%.

**Dataset:** CIFAR-10 — 50,000 training images, 10,000 test images, 32×32×3 (RGB)

In [1]:

!pip install tensorflow matplotlib numpy scipy scikit-learn seaborn

In [2]:

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)

: 

In [ ]:
# loading cifar-10 dataset (already split into train and test)
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

# checking the shape of training and test data
print("Training data shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Test data shape:", x_test.shape)
print("Test labels shape:", y_test.shape)


In [ ]:
# cifar-10 has 10 categories
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# lets visualize some images from the dataset to understand it better
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(x_train[i])
    plt.xlabel(class_names[y_train[i][0]])
plt.suptitle("Sample Images from CIFAR-10 Dataset", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# normalizing pixel values to be between 0 and 1 (originally 0-255)
x_train = x_train / 255.0
x_test = x_test / 255.0

print("Pixel value range after normalization:")
print("Min:", x_train.min(), "Max:", x_train.max())


In [ ]:
# data augmentation - helps the model generalize better and avoid overfitting
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(x_train)

In [ ]:
# building the CNN model with batch normalization and dropout for regularization

model = models.Sequential()

# first conv block - 32 filters
model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)))
model.add(layers.BatchNormalization())
model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.25))

# second conv block - 64 filters
model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.25))

# third conv block - 128 filters
model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.25))

# flattening and dense layers
model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.5))

# output layer - 10 classes with softmax
model.add(layers.Dense(10, activation='softmax'))

model.summary()

In [ ]:
# compiling with adam optimizer
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# using learning rate reducer - reduces lr when accuracy plateaus
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

# training with data augmentation for 30 epochs
history = model.fit(datagen.flow(x_train, y_train, batch_size=64),
                    epochs=30,
                    validation_data=(x_test, y_test),
                    callbacks=[lr_reducer])

In [ ]:
# plotting accuracy and loss curves to check for overfitting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# accuracy plot
ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Training vs Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# loss plot
ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Training vs Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print("\nFinal Training Accuracy: {:.2f}%".format(history.history['accuracy'][-1] * 100))
print("Final Validation Accuracy: {:.2f}%".format(history.history['val_accuracy'][-1] * 100))

if history.history['val_accuracy'][-1] >= history.history['accuracy'][-1]:
    print("\n✅ No overfitting detected - validation accuracy is >= training accuracy")
else:
    gap = history.history['accuracy'][-1] - history.history['val_accuracy'][-1]
    if gap < 0.05:
        print("\n✅ No significant overfitting - gap between train and val is only {:.2f}%".format(gap * 100))
    else:
        print("\n⚠️ Possible overfitting - gap is {:.2f}%".format(gap * 100))

In [ ]:
# evaluating model on test data
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)
print("\nTest Accuracy: {:.2f}%".format(test_accuracy * 100))
print("Test Loss: {:.4f}".format(test_loss))

# making predictions on some test images
predictions = model.predict(x_test[:10])

# visualizing predictions vs actual labels
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(x_test[i])
    predicted_label = class_names[np.argmax(predictions[i])]
    actual_label = class_names[y_test[i][0]]
    color = 'green' if predicted_label == actual_label else 'red'
    plt.xlabel(f"Pred: {predicted_label}\nActual: {actual_label}", color=color, fontsize=9)
plt.suptitle("Model Predictions (Green = Correct, Red = Wrong)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# generating classification report and confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# getting predictions for all test images
y_pred = np.argmax(model.predict(x_test), axis=1)

# classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - CIFAR-10 CNN Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n✅ Final Test Accuracy: {:.2f}% (Target was 85%)".format(test_accuracy * 100))